# UN SOLO CANALE 

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from tensorflow.keras.utils import to_categorical
import os

# Initialize MediaPipe Hand detector
mp_hands = mp.solutions.hands
hands_detector = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.5
)


def create_combined_heatmap(image_rgb, target_size=64, sigma=1.5):
    """
    Create a single heatmap combining all 21 hand keypoints
    
    Args:
        image_rgb: RGB image for MediaPipe
        target_size: output size (e.g., 64)
        sigma: Gaussian spread (DEFAULT 1.5 for clearer keypoints)
    
    Returns:
        combined_heatmap: (target_size, target_size) single channel with all keypoints
        None: if no hand keypoints detected
    """
    results = hands_detector.process(image_rgb)
    
    # ⚠️ RETURN None IF NO KEYPOINTS DETECTED
    if not results.multi_hand_landmarks:
        return None
    
    # Initialize empty heatmap
    heatmap = np.zeros((target_size, target_size), dtype='float32')
    
    hand_landmarks = results.multi_hand_landmarks[0]
    
    # Create coordinate grids once
    x_grid, y_grid = np.meshgrid(np.arange(target_size), np.arange(target_size))
    
    for landmark in hand_landmarks.landmark:
        # Convert normalized coordinates to pixels
        x_px = int(landmark.x * target_size)
        y_px = int(landmark.y * target_size)
        
        # Clamp coordinates to valid range
        x_px = np.clip(x_px, 0, target_size - 1)
        y_px = np.clip(y_px, 0, target_size - 1)
        
        # Add Gaussian blob for this keypoint
        gaussian = np.exp(-((x_grid - x_px)**2 + (y_grid - y_px)**2) / (2 * sigma**2))
        heatmap = np.maximum(heatmap, gaussian)  # Take max to avoid overlapping
    
    return heatmap


def edge_detection(image):
    """Apply adaptive thresholding to enhance hand edges"""
    minValue = 70
    blur = cv2.GaussianBlur(image, (5, 5), 2)
    th3 = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                 cv2.THRESH_BINARY_INV, 11, 2)
    ret, res = cv2.threshold(th3, minValue, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return res


# ============================================================================
# OPTION A: RGB (3 channels) + HEATMAP (1 channel) = 4 CHANNELS TOTAL
# ============================================================================

def preprocess_image_rgb_heatmap(image_path, target_size=64):
    """
    RGB + Heatmap preprocessing → (64, 64, 4)
    
    Output channels:
    - Channels 0-2: RGB image (no edge detection)
    - Channel 3: Combined keypoint heatmap
    
    Args:
        image_path: path to image
        target_size: target dimension
    
    Returns:
        image with shape (64, 64, 4) OR None if no keypoints detected
    """
    try:
        # Load RGB
        img_rgb = cv2.imread(image_path, cv2.IMREAD_COLOR)
        if img_rgb is None:
            return None
        img_rgb = cv2.cvtColor(img_rgb, cv2.COLOR_BGR2RGB)
        
        # Resize with aspect ratio preservation
        h, w = img_rgb.shape[:2]
        aspect_ratio = h / w
        
        if aspect_ratio > 1:
            new_w = int(target_size / aspect_ratio)
            img_resized = cv2.resize(img_rgb, (new_w, target_size))
        else:
            new_h = int(target_size * aspect_ratio)
            img_resized = cv2.resize(img_rgb, (target_size, new_h))
        
        # Create square canvas
        canvas = np.zeros((target_size, target_size, 3), dtype=np.uint8)
        h_resized, w_resized = img_resized.shape[:2]
        y_offset = (target_size - h_resized) // 2
        x_offset = (target_size - w_resized) // 2
        canvas[y_offset:y_offset + h_resized, x_offset:x_offset + w_resized] = img_resized
        
        # Extract combined heatmap
        heatmap = create_combined_heatmap(canvas, target_size)
        
        # ⚠️ RETURN None IF NO KEYPOINTS (excludes image from dataset)
        if heatmap is None:
            return None
        
        # Normalize RGB to [0, 1]
        rgb_normalized = canvas.astype('float32') / 255.0
        
        heatmap = np.expand_dims(heatmap, axis=-1)  # (64, 64, 1)
        
        # Stack: [R, G, B, Heatmap]
        stacked = np.concatenate([rgb_normalized, heatmap], axis=-1)  # (64, 64, 4)
        
        return stacked
        
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None


# ============================================================================
# OPTION B: GRAYSCALE (1 channel) + HEATMAP (1 channel) = 2 CHANNELS TOTAL
# ============================================================================

def preprocess_image_gray_heatmap(image_path, target_size=64):
    """
    Grayscale + Heatmap preprocessing → (64, 64, 2)
    
    Output channels:
    - Channel 0: Edge-detected grayscale
    - Channel 1: Combined keypoint heatmap
    
    Args:
        image_path: path to image
        target_size: target dimension
    
    Returns:
        image with shape (64, 64, 2) OR None if no keypoints detected
    """
    try:
        # Load grayscale for edge detection
        img_gray = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if img_gray is None:
            return None
        
        # Load RGB for MediaPipe
        img_rgb = cv2.imread(image_path, cv2.IMREAD_COLOR)
        img_rgb = cv2.cvtColor(img_rgb, cv2.COLOR_BGR2RGB)
        
        # Apply edge detection
        img_edges = edge_detection(img_gray)
        
        # Resize with aspect ratio preservation
        h, w = img_edges.shape
        aspect_ratio = h / w
        
        if aspect_ratio > 1:
            new_w = int(target_size / aspect_ratio)
            gray_resized = cv2.resize(img_edges, (new_w, target_size))
            rgb_resized = cv2.resize(img_rgb, (new_w, target_size))
        else:
            new_h = int(target_size * aspect_ratio)
            gray_resized = cv2.resize(img_edges, (target_size, new_h))
            rgb_resized = cv2.resize(img_rgb, (target_size, new_h))
        
        # Create square canvas for grayscale
        canvas_gray = np.zeros((target_size, target_size), dtype=np.uint8)
        h_resized, w_resized = gray_resized.shape
        y_offset = (target_size - h_resized) // 2
        x_offset = (target_size - w_resized) // 2
        canvas_gray[y_offset:y_offset + h_resized, x_offset:x_offset + w_resized] = gray_resized
        
        # Create canvas for RGB (for keypoint extraction)
        canvas_rgb = np.zeros((target_size, target_size, 3), dtype=np.uint8)
        canvas_rgb[y_offset:y_offset + h_resized, x_offset:x_offset + w_resized] = rgb_resized
        
        # Extract combined heatmap
        heatmap = create_combined_heatmap(canvas_rgb, target_size)
        
        # ⚠️ RETURN None IF NO KEYPOINTS (excludes image from dataset)
        if heatmap is None:
            return None
        
        # Normalize grayscale to [0, 1]
        gray_normalized = canvas_gray.astype('float32') / 255.0
        gray_channel = np.expand_dims(gray_normalized, axis=-1)  # (64, 64, 1)
        
        heatmap_channel = np.expand_dims(heatmap, axis=-1)  # (64, 64, 1)
        
        # Stack: [Grayscale, Heatmap]
        stacked = np.concatenate([gray_channel, heatmap_channel], axis=-1)  # (64, 64, 2)
        
        return stacked
        
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None


# ============================================================================
# LOADER FUNCTION - CHOOSE YOUR OPTION
# ============================================================================

def load_images_from_folders_with_heatmap(train_path, test_path, target_size=64, mode='grayscale'):
    """
    Load images with heatmaps
    
    Args:
        train_path: training folder path
        test_path: test folder path
        target_size: image size (default 64)
        mode: 'grayscale' for 2 channels or 'rgb' for 4 channels
    
    Returns:
        x_train, x_test with shape (..., 64, 64, 2) or (..., 64, 64, 4)
        y_train, y_test with one-hot encoding
    """
    x_train = []
    y_train = []
    x_test = []
    y_test = []
    
    # Counters for excluded images
    excluded_train = 0
    excluded_test = 0
    
    # Choose preprocessing function
    if mode == 'rgb':
        preprocess_fn = preprocess_image_rgb_heatmap
        expected_channels = 4
        print("🎨 Mode: RGB + Heatmap (4 channels)")
    else:  # grayscale
        preprocess_fn = preprocess_image_gray_heatmap
        expected_channels = 2
        print("⚫ Mode: Grayscale + Heatmap (2 channels)")
    
    character_folders = sorted([d for d in os.listdir(train_path) 
                                if os.path.isdir(os.path.join(train_path, d))])
    
    print(f"Found {len(character_folders)} character classes: {character_folders}")
    print("⚠️  Images without detected keypoints will be EXCLUDED from dataset")
    print("📍 Using sigma=1.5 for precise keypoint localization")
    print("\n" + "="*80)
    
    for class_idx, char_folder in enumerate(character_folders):
        train_char_path = os.path.join(train_path, char_folder)
        test_char_path = os.path.join(test_path, char_folder)
        
        # Load training images
        train_count = 0
        train_excluded = 0
        if os.path.exists(train_char_path):
            for img_file in os.listdir(train_char_path):
                if img_file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                    img_path = os.path.join(train_char_path, img_file)
                    img = preprocess_fn(img_path, target_size)
                    if img is not None and img.shape == (target_size, target_size, expected_channels):
                        x_train.append(img)
                        y_train.append(class_idx)
                        train_count += 1
                    elif img is None:
                        train_excluded += 1
        
        # Load test images
        test_count = 0
        test_excluded = 0
        if os.path.exists(test_char_path):
            for img_file in os.listdir(test_char_path):
                if img_file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                    img_path = os.path.join(test_char_path, img_file)
                    img = preprocess_fn(img_path, target_size)
                    if img is not None and img.shape == (target_size, target_size, expected_channels):
                        x_test.append(img)
                        y_test.append(class_idx)
                        test_count += 1
                    elif img is None:
                        test_excluded += 1
        
        excluded_train += train_excluded
        excluded_test += test_excluded
        
        status = ""
        if train_excluded > 0 or test_excluded > 0:
            status = f" | ❌ Excluded: Train={train_excluded}, Test={test_excluded}"
        
        print(f"Class {class_idx:2d} ({char_folder}): Train={train_count:4d} | Test={test_count:4d}{status}")
    
    print("="*80)
    
    if len(x_train) == 0 or len(x_test) == 0:
        raise ValueError("No images found! Check folder paths.")
    
    x_train = np.array(x_train)
    x_test = np.array(x_test)
    y_train = np.array(y_train)
    y_test = np.array(y_test)
    
    print(f"\n✓ Training set: {x_train.shape} images")
    print(f"✓ Test set: {x_test.shape} images")
    print(f"✓ Classes: {len(np.unique(y_train))}")
    
    if excluded_train > 0 or excluded_test > 0:
        print(f"\n⚠️  Total excluded (no keypoints): Train={excluded_train}, Test={excluded_test}")
    
    num_classes = len(character_folders)
    y_train = to_categorical(y_train, num_classes=num_classes)
    y_test = to_categorical(y_test, num_classes=num_classes)
    
    print(f"✓ Labels one-hot encoded: {y_train.shape}")
    
    return x_train, x_test, y_train, y_test


# ============================================================================

# TRE CANALI


In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from tensorflow.keras.utils import to_categorical
import os

# Initialize MediaPipe Hand detector
mp_hands = mp.solutions.hands
hands_detector = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.5
)


def create_combined_heatmap(image_rgb, target_size=64, sigma=1.0):
    """
    Create 3 separate heatmaps for different hand parts
    
    Args:
        image_rgb: RGB image for MediaPipe
        target_size: output size (e.g., 64)
        sigma: Gaussian spread (default 1.0 for balanced keypoints)
    
    Returns:
        heatmaps: (target_size, target_size, 3) with channels:
                  - Channel 0: Thumb (landmarks 1-4)
                  - Channel 1: Fingers (landmarks 5-20)
                  - Channel 2: Palm/Wrist (landmark 0)
        None: if no hand keypoints detected
    """
    results = hands_detector.process(image_rgb)
    
    # ⚠️ RETURN None IF NO KEYPOINTS DETECTED
    if not results.multi_hand_landmarks:
        return None
    
    # Initialize 3 separate heatmaps
    heatmap_thumb = np.zeros((target_size, target_size), dtype='float32')
    heatmap_fingers = np.zeros((target_size, target_size), dtype='float32')
    heatmap_palm = np.zeros((target_size, target_size), dtype='float32')
    
    hand_landmarks = results.multi_hand_landmarks[0]
    
    # Create coordinate grids once
    x_grid, y_grid = np.meshgrid(np.arange(target_size), np.arange(target_size))
    
    for idx, landmark in enumerate(hand_landmarks.landmark):
        # Convert normalized coordinates to pixels
        x_px = int(landmark.x * target_size)
        y_px = int(landmark.y * target_size)
        
        # Clamp coordinates to valid range
        x_px = np.clip(x_px, 0, target_size - 1)
        y_px = np.clip(y_px, 0, target_size - 1)
        
        # Add Gaussian blob for this keypoint
        gaussian = np.exp(-((x_grid - x_px)**2 + (y_grid - y_px)**2) / (2 * sigma**2))
        
        # Distribute keypoints to appropriate heatmap
        if idx == 0:  # Wrist/Palm base
            heatmap_palm = np.maximum(heatmap_palm, gaussian)
        elif 1 <= idx <= 4:  # Thumb (CMC, MCP, IP, TIP)
            heatmap_thumb = np.maximum(heatmap_thumb, gaussian)
        else:  # Fingers (5-20: index, middle, ring, pinky)
            heatmap_fingers = np.maximum(heatmap_fingers, gaussian)
    
    # Stack into 3 channels
    heatmaps = np.stack([heatmap_thumb, heatmap_fingers, heatmap_palm], axis=-1)
    
    return heatmaps


def edge_detection(image):
    """Apply adaptive thresholding to enhance hand edges"""
    minValue = 70
    blur = cv2.GaussianBlur(image, (5, 5), 2)
    th3 = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                 cv2.THRESH_BINARY_INV, 11, 2)
    ret, res = cv2.threshold(th3, minValue, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return res


# ============================================================================
# OPTION A: RGB (3 channels) + HEATMAP (3 channels) = 6 CHANNELS TOTAL
# ============================================================================

def preprocess_image_rgb_heatmap(image_path, target_size=64, heatmap_weight=1.0):
    """
    RGB + Multi-channel Heatmap preprocessing → (64, 64, 6)
    
    Output channels:
    - Channels 0-2: RGB image
    - Channel 3: Thumb heatmap
    - Channel 4: Fingers heatmap
    - Channel 5: Palm heatmap
    
    Args:
        image_path: path to image
        target_size: target dimension
        heatmap_weight: multiplier to boost heatmap influence (default 1.0)
    
    Returns:
        image with shape (64, 64, 6) OR None if no keypoints detected
    """
    try:
        # Load RGB
        img_rgb = cv2.imread(image_path, cv2.IMREAD_COLOR)
        if img_rgb is None:
            return None
        img_rgb = cv2.cvtColor(img_rgb, cv2.COLOR_BGR2RGB)
        
        # Resize with aspect ratio preservation
        h, w = img_rgb.shape[:2]
        aspect_ratio = h / w
        
        if aspect_ratio > 1:
            new_w = int(target_size / aspect_ratio)
            img_resized = cv2.resize(img_rgb, (new_w, target_size))
        else:
            new_h = int(target_size * aspect_ratio)
            img_resized = cv2.resize(img_rgb, (target_size, new_h))
        
        # Create square canvas
        canvas = np.zeros((target_size, target_size, 3), dtype=np.uint8)
        h_resized, w_resized = img_resized.shape[:2]
        y_offset = (target_size - h_resized) // 2
        x_offset = (target_size - w_resized) // 2
        canvas[y_offset:y_offset + h_resized, x_offset:x_offset + w_resized] = img_resized
        
        # Extract 3-channel heatmap
        heatmaps = create_combined_heatmap(canvas, target_size)
        
        # ⚠️ RETURN None IF NO KEYPOINTS (excludes image from dataset)
        if heatmaps is None:
            return None
        
        # Apply heatmap weight boost if specified
        if heatmap_weight != 1.0:
            heatmaps = np.clip(heatmaps * heatmap_weight, 0, 1)
        
        # Normalize RGB to [0, 1]
        rgb_normalized = canvas.astype('float32') / 255.0
        
        # Stack: [R, G, B, Thumb, Fingers, Palm]
        stacked = np.concatenate([rgb_normalized, heatmaps], axis=-1)  # (64, 64, 6)
        
        return stacked
        
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None


# ============================================================================
# OPTION B: GRAYSCALE (1 channel) + HEATMAP (1 channel) = 2 CHANNELS TOTAL
# ============================================================================

def preprocess_image_gray_heatmap(image_path, target_size=64):
    """
    Grayscale + Heatmap preprocessing → (64, 64, 2)
    
    Output channels:
    - Channel 0: Edge-detected grayscale
    - Channel 1: Combined keypoint heatmap (all 3 heatmaps merged)
    
    Args:
        image_path: path to image
        target_size: target dimension
    
    Returns:
        image with shape (64, 64, 2) OR None if no keypoints detected
    """
    try:
        # Load grayscale for edge detection
        img_gray = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if img_gray is None:
            return None
        
        # Load RGB for MediaPipe
        img_rgb = cv2.imread(image_path, cv2.IMREAD_COLOR)
        img_rgb = cv2.cvtColor(img_rgb, cv2.COLOR_BGR2RGB)
        
        # Apply edge detection
        img_edges = edge_detection(img_gray)
        
        # Resize with aspect ratio preservation
        h, w = img_edges.shape
        aspect_ratio = h / w
        
        if aspect_ratio > 1:
            new_w = int(target_size / aspect_ratio)
            gray_resized = cv2.resize(img_edges, (new_w, target_size))
            rgb_resized = cv2.resize(img_rgb, (new_w, target_size))
        else:
            new_h = int(target_size * aspect_ratio)
            gray_resized = cv2.resize(img_edges, (target_size, new_h))
            rgb_resized = cv2.resize(img_rgb, (target_size, new_h))
        
        # Create square canvas for grayscale
        canvas_gray = np.zeros((target_size, target_size), dtype=np.uint8)
        h_resized, w_resized = gray_resized.shape
        y_offset = (target_size - h_resized) // 2
        x_offset = (target_size - w_resized) // 2
        canvas_gray[y_offset:y_offset + h_resized, x_offset:x_offset + w_resized] = gray_resized
        
        # Create canvas for RGB (for keypoint extraction)
        canvas_rgb = np.zeros((target_size, target_size, 3), dtype=np.uint8)
        canvas_rgb[y_offset:y_offset + h_resized, x_offset:x_offset + w_resized] = rgb_resized
        
        # Extract 3-channel heatmap
        heatmaps = create_combined_heatmap(canvas_rgb, target_size)
        
        # ⚠️ RETURN None IF NO KEYPOINTS (excludes image from dataset)
        if heatmaps is None:
            return None
        
        # Merge 3 heatmaps into single channel by taking maximum
        heatmap = np.max(heatmaps, axis=-1)
        
        # Normalize grayscale to [0, 1]
        gray_normalized = canvas_gray.astype('float32') / 255.0
        gray_channel = np.expand_dims(gray_normalized, axis=-1)  # (64, 64, 1)
        
        heatmap_channel = np.expand_dims(heatmap, axis=-1)  # (64, 64, 1)
        
        # Stack: [Grayscale, Heatmap]
        stacked = np.concatenate([gray_channel, heatmap_channel], axis=-1)  # (64, 64, 2)
        
        return stacked
        
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None


# ============================================================================
# LOADER FUNCTION - CHOOSE YOUR OPTION
# ============================================================================

def load_images_from_folders_with_heatmap(train_path, test_path, target_size=64, mode='grayscale', heatmap_weight=1.0):
    """
    Load images with heatmaps
    
    Args:
        train_path: training folder path
        test_path: test folder path
        target_size: image size (default 64)
        mode: 'grayscale' for 2 channels or 'rgb' for 6 channels
        heatmap_weight: multiplier for heatmap intensity (default 1.0, only for RGB mode)
    
    Returns:
        x_train, x_test with shape (..., 64, 64, 2) or (..., 64, 64, 6)
        y_train, y_test with one-hot encoding
    """
    x_train = []
    y_train = []
    x_test = []
    y_test = []
    
    # Counters for excluded images
    excluded_train = 0
    excluded_test = 0
    
    # Choose preprocessing function
    if mode == 'rgb':
        preprocess_fn = lambda path: preprocess_image_rgb_heatmap(path, target_size, heatmap_weight)
        expected_channels = 6
        print("🎨 Mode: RGB + Multi-Heatmap (6 channels)")
        print(f"   └─ Channels: RGB (3) + Thumb/Fingers/Palm (3)")
        if heatmap_weight != 1.0:
            print(f"   └─ Heatmap boost: {heatmap_weight}x")
    else:  # grayscale
        preprocess_fn = lambda path: preprocess_image_gray_heatmap(path, target_size)
        expected_channels = 2
        print("⚫ Mode: Grayscale + Heatmap (2 channels)")
    
    character_folders = sorted([d for d in os.listdir(train_path) 
                                if os.path.isdir(os.path.join(train_path, d))])
    
    print(f"Found {len(character_folders)} character classes: {character_folders}")
    print("⚠️  Images without detected keypoints will be EXCLUDED from dataset")
    print(f"📍 Using sigma=1.0 for keypoint localization")
    print("\n" + "="*80)
    
    for class_idx, char_folder in enumerate(character_folders):
        train_char_path = os.path.join(train_path, char_folder)
        test_char_path = os.path.join(test_path, char_folder)
        
        # Load training images
        train_count = 0
        train_excluded = 0
        if os.path.exists(train_char_path):
            for img_file in os.listdir(train_char_path):
                if img_file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                    img_path = os.path.join(train_char_path, img_file)
                    img = preprocess_fn(img_path)
                    if img is not None and img.shape == (target_size, target_size, expected_channels):
                        x_train.append(img)
                        y_train.append(class_idx)
                        train_count += 1
                    elif img is None:
                        train_excluded += 1
        
        # Load test images
        test_count = 0
        test_excluded = 0
        if os.path.exists(test_char_path):
            for img_file in os.listdir(test_char_path):
                if img_file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                    img_path = os.path.join(test_char_path, img_file)
                    img = preprocess_fn(img_path)
                    if img is not None and img.shape == (target_size, target_size, expected_channels):
                        x_test.append(img)
                        y_test.append(class_idx)
                        test_count += 1
                    elif img is None:
                        test_excluded += 1
        
        excluded_train += train_excluded
        excluded_test += test_excluded
        
        status = ""
        if train_excluded > 0 or test_excluded > 0:
            status = f" | ❌ Excluded: Train={train_excluded}, Test={test_excluded}"
        
        print(f"Class {class_idx:2d} ({char_folder}): Train={train_count:4d} | Test={test_count:4d}{status}")
    
    print("="*80)
    
    if len(x_train) == 0 or len(x_test) == 0:
        raise ValueError("No images found! Check folder paths.")
    
    x_train = np.array(x_train)
    x_test = np.array(x_test)
    y_train = np.array(y_train)
    y_test = np.array(y_test)
    
    print(f"\n✓ Training set: {x_train.shape} images")
    print(f"✓ Test set: {x_test.shape} images")
    print(f"✓ Classes: {len(np.unique(y_train))}")
    
    if excluded_train > 0 or excluded_test > 0:
        print(f"\n⚠️  Total excluded (no keypoints): Train={excluded_train}, Test={excluded_test}")
    
    num_classes = len(character_folders)
    y_train = to_categorical(y_train, num_classes=num_classes)
    y_test = to_categorical(y_test, num_classes=num_classes)
    
    print(f"✓ Labels one-hot encoded: {y_train.shape}")
    
    return x_train, x_test, y_train, y_test


# ============================================================================